In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
data_clients = pd.read_excel("../data/default_of_credit_card_clients.xls")
data_clients.head(5)


In [ ]:
#forma
data_clients.shape

In [ ]:
data_clients.info()

In [ ]:
data_clients.describe()

In [ ]:
#valores nulos
data_clients.isnull().sum()

In [ ]:
#valores duplicados
print(data_clients.duplicated().sum())
print("__________________________________________________________________________________________________")

In [ ]:
data_clients.columns = data_clients.iloc[0]
data_clients = data_clients.drop(0)
data_clients.columns

In [ ]:
# Revisando el rango de datos de las variables y agrupando
for c in ["SEX","EDUCATION","MARRIAGE"]:
    print(c, data_clients[c].unique())

pays = [c for c in data_clients.columns if c.startswith("PAY_")]
for c in pays:
    print(c, sorted(data_clients[c].unique()))

In [ ]:
data_clients["TOTAL_BILL"] = data_clients[["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6"]].sum(axis=1)
data_clients["TOTAL_PAY"] = data_clients[["PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]].sum(axis=1)
data_clients["DEBT_RATIO"] = data_clients["TOTAL_BILL"] / data_clients["LIMIT_BAL"]           # que tanto usa del límite
data_clients["%_RATIO_PAYT"] = data_clients["TOTAL_PAY"] / data_clients["TOTAL_BILL"].replace(0, np.nan)  # qué % de la deuda es la que paga
data_clients["AVG_BILL"] = data_clients[["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6"]].mean(axis=1)
data_clients["MAX_DEL_PAY"] = data_clients[["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]].max(axis=1)   # máximo retraso registrado

In [ ]:
data_clients.head(5)

In [ ]:
#ahora vamos a determinr los valores atípicos
def detect_outliers(series):

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (series < lower) | (series > upper)

    return outliers.sum(), lower, upper

In [ ]:
variables = ["LIMIT_BAL","TOTAL_BILL","TOTAL_PAY","DEBT_RATIO","AVG_BILL"]

for var in variables:
    
    count, low, high = detect_outliers(data_clients[var].dropna())

    print(f"{var}")
    print("outliers:", count)
    print("lower bound:", low)
    print("upper bound:", high)
    print("-------------")

In [ ]:
#ahora exportamos la base de datos limpia
data_clients.to_csv("../data/credit_clients_clean.csv", index=False)